In [ ]:
# ==============================================================================
# PROJEK UAS MPML - INTELLIGENT REGIONAL ANALYTICS SYSTEM (IRAS) JATENG 2024
# METODE: SUPERVISED, UNSUPERVISED & ENSEMBLE LEARNING (STUNTING & KEMISKINAN)
# NIM: 24611028 | NAMA: REVA RAHMADDANI
# DOSEN PENGUJI: Dr. RB Fajriya Hakim, M.Si.
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
import warnings

warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, silhouette_score
)
from scipy.cluster.hierarchy import dendrogram, linkage

# Konfigurasi Tampilan Plot
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10
RANDOM_STATE = 42

print("==============================================================================")
print("CHUNK 1: DATA AKUISISI & INTEGRASI (SOAL NO. 1)")
print("==============================================================================")

file_path_colab = "/content/Dataset_Kemiskinan_Stunting_JawaTengah_2024.xlsx"
file_path_local = r"d:\KULIAH\Semester 4\MPML\24611028_UAS MPML\Dataset\Dataset_Kemiskinan_Stunting_JawaTengah_2024.xlsx"
file_path_direct = r"Dataset/Dataset_Kemiskinan_Stunting_JawaTengah_2024.xlsx"

if os.path.exists(file_path_direct):
    file_path = file_path_direct
elif os.path.exists(file_path_local):
    file_path = file_path_local
elif os.path.exists(file_path_colab):
    file_path = file_path_colab
else:
    file_path = "Dataset_Kemiskinan_Stunting_JawaTengah_2024.xlsx"

df = pd.read_excel(file_path, sheet_name='Data_Jateng_2024')

# Standardisasi Nama Kolom Internal
df.columns = [
    "kode_wilayah", "kabupaten_kota", "persentase_penduduk_miskin",
    "prevalensi_stunting", "ipm", "pengeluaran_per_kapita",
    "akses_sanitasi_layak", "akses_air_minum_layak",
    "jumlah_puskesmas", "curah_hujan_tahunan"
]
df.columns = df.columns.str.strip()

print("=== DATASET OVERVIEW ===")
print(f"Jumlah Observasi: {df.shape[0]} Wilayah (29 Kabupaten + 6 Kota)")
print(f"Jumlah Kolom: {df.shape[1]}")

assert df.shape[0] == 35, f"Data tidak lengkap! Hanya {df.shape[0]} dari 35 wilayah termuat."
print("[SUCCESS] Data berhasil diverifikasi lengkap (35 Kabupaten/Kota Jawa Tengah).")
print(df.head(5).to_string(index=False))

In [ ]:
print("\n==============================================================================")
print("CHUNK 2: DATA PREPARATION & PREPROCESSING (SOAL NO. 2)")
print("==============================================================================")

poverty_col = "persentase_penduduk_miskin"
feature_cols_all = [
    "prevalensi_stunting", "ipm", "pengeluaran_per_kapita",
    "akses_sanitasi_layak", "akses_air_minum_layak",
    "jumlah_puskesmas", "curah_hujan_tahunan"
]

# --- 2a. Missing Value Check ---
print("=== 2a. MISSING VALUE ===")
missing_total = df[feature_cols_all + [poverty_col]].isnull().sum().sum()
print(f"Total sel kosong: {missing_total} dari {len(feature_cols_all)+1} variabel x 35 wilayah")

# --- 2b. Deteksi Outlier (Metode IQR) ---
print("\n=== 2b. DETEKSI OUTLIER (IQR METHOD) ===")
def cek_outlier_iqr(data, cols):
    hasil = {}
    for col in cols:
        Q1, Q3 = data[col].quantile([0.25, 0.75])
        IQR = Q3 - Q1
        lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR
        out = data[(data[col] < lo) | (data[col] > hi)]
        if len(out) > 0:
            hasil[col] = out[["kabupaten_kota", col]].values.tolist()
    return hasil

outlier_report = cek_outlier_iqr(df, feature_cols_all)
if outlier_report:
    for col, rows in outlier_report.items():
        print(f"Outlier pada '{col}': {rows}")
else:
    print("Tidak ditemukan outlier ekstrem melebihi batas 1.5*IQR.")

# --- 2c. Transformasi Variabel (Log Transform untuk Skewness Tinggi) ---
print("\n=== 2c. TRANSFORMASI VARIABEL (SKEWNESS HANDLING) ===")
skew_before = df[feature_cols_all].skew().sort_values(ascending=False)
print("Skewness sebelum transformasi:\n", skew_before.round(3))

skewed_cols = skew_before[skew_before.abs() > 1.0].index.tolist()
print(f"\nVariabel dengan |skewness| > 1.0 (di-log-transform): {skewed_cols}")

feature_cols_final_pool = []
for col in feature_cols_all:
    if col in skewed_cols:
        new_col = f"log_{col}"
        df[new_col] = np.log1p(df[col])
        feature_cols_final_pool.append(new_col)
    else:
        feature_cols_final_pool.append(col)

print("\nSkewness sesudah transformasi:\n", df[feature_cols_final_pool].skew().sort_values(ascending=False).round(3))

# --- 2d. Formulasi Variabel Target (Binary Classification) ---
print("\n=== 2d. TARGET VARIABLE FORMULATION ===")
median_poverty = df[poverty_col].median()
df["Target_Kemiskinan"] = (df[poverty_col] >= median_poverty).astype(int)
df["Target_Label"] = df["Target_Kemiskinan"].map({1: "Kemiskinan Tinggi", 0: "Kemiskinan Rendah"})
df["tipe_wilayah"] = df["kabupaten_kota"].apply(lambda x: "Kota" if x.startswith("Kota") else "Kabupaten")

print(f"Median Persentase Kemiskinan Jateng 2024: {median_poverty:.2f}%")
print("Distribusi Kelas Target:")
print(df["Target_Label"].value_counts())

In [ ]:
print("\n==============================================================================")
print("CHUNK 3: EXPLORATORY DATA ANALYSIS (SOAL NO. 3)")
print("==============================================================================")

print("\n=== STATISTIK DESKRIPTIF (SELEKSI VARIABEL TERTRANSFORMASI) ===")
print(df[feature_cols_final_pool + [poverty_col]].describe().T.round(2))

# 3a. Histogram Subplots 4x2
fig, axes = plt.subplots(4, 2, figsize=(14, 12))
for ax, col in zip(axes.flatten(), feature_cols_final_pool):
    sns.histplot(df[col], kde=True, ax=ax, color="teal")
    ax.set_title(f"Distribusi {col}", fontsize=10, fontweight="bold")
for ax in axes.flatten()[len(feature_cols_final_pool):]:
    ax.axis("off")
plt.tight_layout()
plt.savefig("eda_distribusi.png", dpi=300)
plt.close()
print("[SUCCESS] Grafik distribusi tersimpan: eda_distribusi.png")

# 3b. Correlation Heatmap
plt.figure(figsize=(10, 8))
corr_matrix = df[feature_cols_final_pool + [poverty_col]].corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.5)
plt.title("Matriks Korelasi Pearson - Indikator IRAS Jawa Tengah 2024", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("heatmap_korelasi.png", dpi=300)
plt.close()
print("[SUCCESS] Heatmap korelasi tersimpan: heatmap_korelasi.png")

# 3c. Scatterplot IPM vs Kemiskinan
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="ipm", y=poverty_col, hue="Target_Label", style="tipe_wilayah", s=100, palette={"Kemiskinan Tinggi": "crimson", "Kemiskinan Rendah": "seagreen"})
plt.axhline(y=median_poverty, color="black", linestyle="--", alpha=0.7, label=f"Median ({median_poverty:.2f}%)")
plt.title("Scatter Plot: IPM vs Persentase Penduduk Miskin (Jateng 2024)", fontsize=12, fontweight="bold")
plt.xlabel("Indeks Pembangunan Manusia (IPM)")
plt.ylabel("Persentase Penduduk Miskin (%)")
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig("scatter_ipm_kemiskinan.png", dpi=300)
plt.close()
print("[SUCCESS] Scatter plot IPM vs Kemiskinan tersimpan: scatter_ipm_kemiskinan.png")

In [ ]:
print("\n==============================================================================")
print("CHUNK 4: FEATURE SELECTION & MULTICOLLINEARITY (SOAL NO. 2, BAGIAN AKHIR)")
print("==============================================================================")

corr_matrix_pred = df[feature_cols_final_pool].corr().abs()
upper = corr_matrix_pred.where(np.triu(np.ones(corr_matrix_pred.shape), k=1).astype(bool))
high_corr_pairs = upper.stack().sort_values(ascending=False)
high_corr_pairs_filtered = high_corr_pairs[high_corr_pairs > 0.85]

print("\n=== PASANGAN VARIABEL KORELASI TINGGI (>0.85) ===")
if len(high_corr_pairs_filtered) > 0:
    print(high_corr_pairs_filtered.round(3))
else:
    print("Tidak ada pasangan variabel prediktor dengan korelasi ekstrem > 0.85.")

# --- Penanganan multikolinearitas: BUKAN cuma deteksi, tapi benar-benar membuang salah satu ---
# Aturan: untuk tiap pasangan variabel dengan korelasi > 0.85, pertahankan variabel yang
# korelasinya LEBIH KUAT terhadap target (poverty_col), buang yang lebih lemah.
# Ini otomatis konsisten dengan analisis: IPM (r=-0.687 ke target) dipertahankan,
# Pengeluaran per Kapita (r=-0.636 ke target) dibuang -- karena IPM juga secara konsep
# sudah menyerap dimensi pengeluaran (komponen resmi IPM), dan Persentase Penduduk Miskin
# secara definisi BPS dihitung dari distribusi pengeluaran per kapita itu sendiri --
# sehingga mempertahankan keduanya berisiko kuasi-sirkularitas dengan target.
kolom_dibuang = []
for (var1, var2), corr_val in high_corr_pairs_filtered.items():
    r1 = abs(df[var1].corr(df[poverty_col]))
    r2 = abs(df[var2].corr(df[poverty_col]))
    var_dibuang, var_dipertahankan = (var1, var2) if r1 < r2 else (var2, var1)
    if var_dibuang not in kolom_dibuang:
        kolom_dibuang.append(var_dibuang)
    print(f"\nPasangan '{var1}' vs '{var2}' (r={corr_val:.3f}):")
    print(f"  Korelasi ke target -> {var1}: {r1:.3f} | {var2}: {r2:.3f}")
    print(f"  Dipertahankan: '{var_dipertahankan}' (korelasi ke target lebih kuat)")
    print(f"  Dibuang      : '{var_dibuang}'")

selected_features = [c for c in feature_cols_final_pool if c not in kolom_dibuang]
print(f"\nJumlah variabel awal: {len(feature_cols_final_pool)} -> Final Prediktor: {len(selected_features)}")
print("Variabel dibuang (multikolinearitas):", kolom_dibuang)
print("Variabel prediktor final:", selected_features)

In [ ]:
print("\n==============================================================================")
print("CHUNK 5: TRAIN-TEST SPLIT & STANDARDISASI (SOAL NO. 4)")
print("==============================================================================")

X = df[selected_features]
y = df["Target_Kemiskinan"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_scaled_all = scaler.fit_transform(X)

print(f"Train set: {X_train.shape[0]} wilayah | Test set: {X_test.shape[0]} wilayah")

In [ ]:
print("\n==============================================================================")
print("CHUNK 6: SUPERVISED LEARNING - 5 MODEL TUNGGAL (SOAL NO. 4)")
print("==============================================================================")

models_single = {
    "KNN": KNeighborsClassifier(n_neighbors=3),
    "SVM": SVC(kernel='rbf', random_state=RANDOM_STATE, probability=True),
    "Decision Tree": DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE),
    "Naive Bayes": GaussianNB(),
    "ANN": MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=2000, random_state=RANDOM_STATE)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results_single = []

for name, model in models_single.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, "predict_proba") else y_pred
    cv_scores = cross_val_score(model, X_scaled_all, y, cv=cv, scoring="accuracy")

    results_single.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
        "CV_Mean_Accuracy": cv_scores.mean(),
        "CV_Std": cv_scores.std(),
    })

df_res_single = pd.DataFrame(results_single)
print("\n=== HASIL EVALUASI 5 MODEL KLASIFIKASI TUNGGAL ===")
print(df_res_single.round(3).to_string(index=False))


In [ ]:
print("\n==============================================================================")
print("CHUNK 7: UNSUPERVISED LEARNING - CLUSTERING (SOAL NO. 5)")
print("==============================================================================")

sil_scores = []
K_range = range(2, 7)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled_all)
    sil_scores.append(silhouette_score(X_scaled_all, labels))

plt.figure(figsize=(8, 4))
plt.plot(list(K_range), sil_scores, marker="o", color="navy", linewidth=2)
plt.title("Silhouette Score vs Jumlah Cluster (k)", fontsize=12, fontweight="bold")
plt.xlabel("Jumlah Cluster (k)"); plt.ylabel("Silhouette Score")
plt.tight_layout()
plt.savefig("silhouette_pemilihan_k.png", dpi=300)
plt.close()

k_optimal = list(K_range)[int(np.argmax(sil_scores))]
print(f"Optimal cluster k berdasarkan Silhouette Score tertinggi: k={k_optimal} (Score: {max(sil_scores):.4f})")

# Dendrogram Hierarchical Clustering
plt.figure(figsize=(12, 6))
linked = linkage(X_scaled_all, method='ward')
dendrogram(linked, labels=df["kabupaten_kota"].values, orientation='top', distance_sort='descending')
plt.title("Dendrogram Hierarchical Clustering 35 Kab/Kota Jateng 2024", fontsize=13, fontweight="bold")
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("dendrogram_hierarchical_jateng.png", dpi=300)
plt.close()

# K-Means Final
kmeans_opt = KMeans(n_clusters=k_optimal, random_state=RANDOM_STATE, n_init=10)
df["Cluster"] = kmeans_opt.fit_predict(X_scaled_all)

# --- Penamaan cluster OTOMATIS berdasarkan rata-rata persentase kemiskinan tiap cluster ---
# (bukan hardcode "0=Sejahtera, 1=Rentan" -- karena nomor cluster dari KMeans itu ACAK,
#  urutannya bisa terbalik setiap kali data/parameter berubah)
cluster_poverty_rank = df.groupby("Cluster")[poverty_col].mean().sort_values()
ordinal_labels = [f"Cluster {i} ({tag})" for i, tag in enumerate(
    ["Paling Sejahtera"] + ["Menengah"] * max(0, len(cluster_poverty_rank) - 2) + ["Paling Rentan"]
    if len(cluster_poverty_rank) > 1 else ["Tunggal"]
)]
cluster_names_map = dict(zip(cluster_poverty_rank.index, ordinal_labels))
df["Cluster_Name"] = df["Cluster"].map(cluster_names_map)

print("\nPemetaan label cluster (divalidasi otomatis dari rata-rata % kemiskinan):")
for cl, label in cluster_names_map.items():
    print(f"  Cluster {cl} -> {label} (rata-rata kemiskinan: {cluster_poverty_rank[cl]:.2f}%)")

print("\n=== PROFIL RATA-RATA TIAP CLUSTER ===")
cluster_profile = df.groupby("Cluster")[selected_features + [poverty_col]].mean().T
print(cluster_profile.round(2))

print("\n=== CROSSTAB: CLUSTER vs KATEGORI KEMISKINAN ===")
print(pd.crosstab(df["Cluster_Name"], df["Target_Label"]))

In [ ]:
print("\n==============================================================================")
print("CHUNK 8: ENSEMBLE LEARNING (SOAL NO. 6)")
print("==============================================================================")

models_ensemble = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    "AdaBoost": AdaBoostClassifier(n_estimators=50, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
}

results_ensemble = []
for name, model in models_ensemble.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    cv_scores = cross_val_score(model, X_scaled_all, y, cv=cv, scoring="accuracy")
    results_ensemble.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
        "CV_Mean_Accuracy": cv_scores.mean(),
        "CV_Std": cv_scores.std(),
    })

df_res_ensemble = pd.DataFrame(results_ensemble)
print("\n=== HASIL EVALUASI MODEL ENSEMBLE ===")
print(df_res_ensemble.round(3).to_string(index=False))

df_all_models = pd.concat([df_res_single, df_res_ensemble], ignore_index=True)
df_all_models_ranked = df_all_models.sort_values("CV_Mean_Accuracy", ascending=False)
print("\n=== RANKING SELURUH MODEL MACHINE LEARNING (Berdasarkan CV Mean Accuracy) ===")
print(df_all_models_ranked[["Model", "Accuracy", "F1-Score", "CV_Mean_Accuracy", "CV_Std"]].round(3).to_string(index=False))



In [ ]:
print("\n==============================================================================")
print("CHUNK 8b: CLASSIFICATION REPORT TRAINING vs TESTING (DIAGNOSIS OVERFITTING)")
print("==============================================================================")
from sklearn.metrics import classification_report

def get_classification_df(model, X, y, dataset_label, model_name):
    y_pred = model.predict(X)
    report_dict = classification_report(
        y, y_pred, target_names=["Kemiskinan Rendah", "Kemiskinan Tinggi"],
        output_dict=True, zero_division=0
    )
    df_report = pd.DataFrame(report_dict).T.reset_index().rename(columns={"index": "Kelas"})
    df_report.insert(0, "Dataset", dataset_label)
    df_report.insert(0, "Model", model_name)
    return df_report

all_dict_models_full = {**models_single, **models_ensemble}
all_reports = []
for name, model in all_dict_models_full.items():
    model.fit(X_train_scaled, y_train)
    all_reports.append(get_classification_df(model, X_train_scaled, y_train, "Training", name))
    all_reports.append(get_classification_df(model, X_test_scaled, y_test, "Testing", name))

df_all_classification_reports = pd.concat(all_reports, ignore_index=True)
print(f"\nTotal baris dataframe konsolidasi: {df_all_classification_reports.shape[0]} "
      f"({len(all_dict_models_full)} model x 2 dataset x 5 baris kelas/summary)")

# --- Contoh cara "panggil satu-satu" dari dataframe gabungan (ganti nama model & dataset sesuai kebutuhan) ---
for nm in all_dict_models_full.keys():
    print(f"\n--- {nm}: Training vs Testing ---")
    subset = df_all_classification_reports[df_all_classification_reports["Model"] == nm]
    print(subset.round(3).to_string(index=False))

# Cek gap akurasi training vs testing tiap model -- indikasi overfitting
print("\n=== RINGKASAN GAP AKURASI (Training - Testing) ===")
acc_train = df_all_classification_reports[
    (df_all_classification_reports["Kelas"] == "accuracy") & (df_all_classification_reports["Dataset"] == "Training")
].set_index("Model")["precision"]
acc_test = df_all_classification_reports[
    (df_all_classification_reports["Kelas"] == "accuracy") & (df_all_classification_reports["Dataset"] == "Testing")
].set_index("Model")["precision"]
gap_df = pd.DataFrame({"Akurasi_Training": acc_train, "Akurasi_Testing": acc_test})
gap_df["Gap (indikasi overfitting)"] = gap_df["Akurasi_Training"] - gap_df["Akurasi_Testing"]
print(gap_df.sort_values("Gap (indikasi overfitting)", ascending=False).round(3))

In [ ]:
print("\n==============================================================================")
print("CHUNK 9: HYPERPARAMETER TUNING & MODEL EXPORT (ADAPTASI DARI KATING)")
print("==============================================================================")

param_grid_rf = {
    'n_estimators': [50, 100, 150],
    'max_depth': [None, 3, 5],
    'min_samples_split': [2, 5],
    'criterion': ['gini', 'entropy']
}

grid_search_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid=param_grid_rf,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

grid_search_rf.fit(X_train_scaled, y_train)

print("Best Parameters Random Forest:", grid_search_rf.best_params_)
best_rf = grid_search_rf.best_estimator_
models_ensemble["Random Forest"] = best_rf

y_pred_best = best_rf.predict(X_test_scaled)
print(f"Evaluasi Random Forest Tuned - Accuracy: {accuracy_score(y_test, y_pred_best):.3f} | F1-Score: {f1_score(y_test, y_pred_best):.3f}")

joblib.dump(best_rf, "best_model_iras_jateng.pkl")
joblib.dump(scaler, "scaler_iras_jateng.pkl")
joblib.dump(selected_features, "features_iras_jateng.pkl")
print("[SUCCESS] Model, Scaler, dan Feature list berhasil diexport via Joblib!")

In [ ]:
print("\n==============================================================================")
print("CHUNK 10: FEATURE IMPORTANCE ANALYSIS (SOAL NO. 7)")
print("==============================================================================")

importances = best_rf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 5))
sns.barplot(x=importances[indices], y=np.array(selected_features)[indices], palette="magma")
plt.title("Tingkat Kepentingan Variabel (Random Forest Feature Importance)", fontsize=13, fontweight="bold")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.savefig("feature_importance_jateng.png", dpi=300)
plt.close()

print("Urutan Variabel Paling Berpengaruh Terhadap Kemiskinan:")
for i in range(len(selected_features)):
    idx = indices[i]
    print(f"{i+1}. {selected_features[idx]} (Score: {importances[idx]:.4f})")

In [ ]:
print("\n==============================================================================")
print("CHUNK 11: VISUALISASI PERBANDINGAN PERFORMA MODEL & CONFUSION MATRICES (SOAL NO. 8)")
print("==============================================================================")

plt.figure(figsize=(14, 6))
df_melted = df_all_models.melt(
    id_vars="Model",
    value_vars=["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"],
    var_name="Metric", value_name="Score"
)
sns.barplot(data=df_melted, x="Model", y="Score", hue="Metric", palette="viridis")
plt.title("Perbandingan Performa Model Machine Learning IRAS Jawa Tengah", fontsize=14, fontweight="bold")
plt.xticks(rotation=30, ha="right")
plt.ylim(0, 1.1)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig("grafik_perbandingan_model_jateng.png", dpi=300)
plt.close()
print("[SUCCESS] Grafik perbandingan model tersimpan: grafik_perbandingan_model_jateng.png")

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
all_8_models = ["KNN", "SVM", "Decision Tree", "Naive Bayes", "ANN", "Random Forest", "AdaBoost", "Gradient Boosting"]
all_dict_models = {**models_single, **models_ensemble}

for ax, m_name in zip(axes.flatten(), all_8_models):
    m_obj = all_dict_models[m_name]
    m_obj.fit(X_train_scaled, y_train)
    cm = confusion_matrix(y_test, m_obj.predict(X_test_scaled))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False,
                xticklabels=["Rendah", "Tinggi"], yticklabels=["Rendah", "Tinggi"])
    category_type = "Supervised" if m_name in models_single else "Ensemble"
    ax.set_title(f"{m_name}\n({category_type})", fontsize=11, fontweight="bold")
    ax.set_xlabel("Prediksi"); ax.set_ylabel("Aktual")

plt.tight_layout()
plt.savefig("confusion_matrices_all_models.png", dpi=300)
plt.close()
print("[SUCCESS] Confusion Matrices seluruh 8 model tersimpan: confusion_matrices_all_models.png")


In [ ]:
print("\n==============================================================================")
print("CHUNK 12: PEMETAAN SPASIAL -- STATIS & INTERAKTIF (SOAL NO. 8)")
print("==============================================================================")

plt.figure(figsize=(13, 8))
n_cluster_found = df["Cluster"].nunique()
palette_list = sns.color_palette("RdYlGn_r", n_cluster_found)  # merah=rentan (miskin tinggi), hijau=sejahtera
palette = {cl: palette_list[i] for i, cl in enumerate(sorted(df["Cluster"].unique()))}
sns.scatterplot(
    data=df, x="ipm", y=poverty_col, hue="Cluster",
    style="tipe_wilayah", palette=palette, s=200, alpha=0.9, edgecolor="black"
)
for i in range(df.shape[0]):
    row = df.iloc[i]
    plt.text(row["ipm"] + 0.12, row[poverty_col] - 0.08, row["kabupaten_kota"], fontsize=8, alpha=0.85)

plt.axhline(y=median_poverty, color='darkred', linestyle=':', alpha=0.8,
            label=f'Median Kemiskinan ({median_poverty:.2f}%)')
plt.title(f"Pemetaan Tipologi 35 Kabupaten/Kota Jawa Tengah (K-Means, k={k_optimal})", fontsize=13, fontweight="bold")
plt.xlabel("Indeks Pembangunan Manusia (IPM)")
plt.ylabel("Persentase Penduduk Miskin (%)")
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title="Cluster K-Means")
plt.tight_layout()
plt.savefig("pemetaan_cluster_jateng.png", dpi=300, bbox_inches='tight')
plt.close()
print("[SUCCESS] Tipologi spasial tersimpan: pemetaan_cluster_jateng.png")

# --- 12b. Peta geografis interaktif (Folium) ---
coords_jateng = {
    3301: (-7.7279, 109.0060), 3302: (-7.4646, 109.1764), 3303: (-7.3879, 109.3639),
    3304: (-7.3797, 109.6974), 3305: (-7.6706, 109.6600), 3306: (-7.7161, 109.9990),
    3307: (-7.3633, 109.9009), 3308: (-7.4313, 110.2175), 3309: (-7.5360, 110.5959),
    3310: (-7.7058, 110.6019), 3311: (-7.6811, 110.8340), 3312: (-7.8136, 110.9255),
    3313: (-7.6256, 111.0505), 3314: (-7.4277, 110.9351), 3315: (-7.0877, 110.9161),
    3316: (-7.0706, 111.4173), 3317: (-6.7093, 111.3414), 3318: (-6.7548, 111.0381),
    3319: (-6.8048, 110.8405), 3320: (-6.5891, 110.6685), 3321: (-6.8943, 110.6386),
    3322: (-7.2008, 110.4395), 3323: (-7.3168, 110.1691), 3324: (-7.0253, 110.2057),
    3325: (-7.0149, 109.8492), 3326: (-7.0250, 109.6053), 3327: (-7.0425, 109.4319),
    3328: (-7.0287, 109.1415), 3329: (-7.0016, 108.9702), 3371: (-7.4706, 110.2178),
    3372: (-7.5755, 110.8243), 3373: (-7.3305, 110.5084), 3374: (-6.9667, 110.4167),
    3375: (-6.8886, 109.6753), 3376: (-6.8671, 109.1378)
}

def get_lat_lon(kw):
    try:
        code_int = int(round(float(kw) * 100))
    except:
        code_int = 0
    return coords_jateng.get(code_int, (-7.1501, 110.1403))

df['lat'] = df['kode_wilayah'].map(lambda k: get_lat_lon(k)[0])
df['lon'] = df['kode_wilayah'].map(lambda k: get_lat_lon(k)[1])

try:
    import folium
    peta_jateng = folium.Map(location=[-7.1501, 110.1403], zoom_start=9, tiles='OpenStreetMap')
    colors = {cl: '#%02x%02x%02x' % tuple(int(c*255) for c in palette_list[i])
              for i, cl in enumerate(sorted(df["Cluster"].unique()))}

    for idx, row in df.iterrows():
        popup_html = f"<div style='font-family: Arial; width: 200px;'><b>{row['kabupaten_kota']}</b><br>Kategori: {row['Cluster_Name']}</div>"
        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=10, color=colors[row['Cluster']], fill=True, fill_opacity=0.8,
            popup=folium.Popup(popup_html, max_width=250)
        ).add_to(peta_jateng)

    peta_jateng.save('peta_spasial_jateng_interaktif.html')
    print("[SUCCESS] Peta interaktif k=2 berhasil dibuat: peta_spasial_jateng_interaktif.html")
except ImportError:
    print("[INFO] Folium tidak tersedia di lingkungan ini -- peta statis (12a) tetap tersimpan.")


In [ ]:
print("\n==============================================================================")
print("CHUNK 13: PERBANDINGAN RATA-RATA VARIABEL PER CLUSTER")
print("==============================================================================")

comp_df = df.groupby('Cluster_Name')[selected_features + [poverty_col]].mean().T
print("\n=== PERBANDINGAN RATA-RATA VARIABEL PER CLUSTER ===")
print(comp_df.round(4))
if comp_df.shape[1] == 2:
    diff_col = comp_df.columns[1] + " MINUS " + comp_df.columns[0]
    comp_df[diff_col] = comp_df.iloc[:, 1] - comp_df.iloc[:, 0]
    print(f"\nKolom selisih ditambahkan: {diff_col}")
    print(comp_df[[diff_col]].round(4))

print("\n[FINISH] ANALISIS MPML IRAS JAWA TENGAH 2024 SELESAI DENGAN SUKSES!")